## Question

Three tokens start at positions 0, 4, and 8 on a 12-position circular track. At each tick, each token independently moves one position clockwise or counterclockwise, each with probability $1/2$.

What is the probability that all three tokens occupy the same position at least once within 20 ticks?

A simulation gives an estimate. A small state-probability calculation gives the exact answer for this finite time horizon.

## Solution

Track the probability of each ordered triple of positions while the tokens have not yet met. From each such state, there are $2^3=8$ equally likely combinations of moves at the next tick. Add the probability of transitions where all three positions agree to the probability of having met; carry all other probability forward to the next tick.

Applying this update for 20 ticks gives a probability of approximately 21.686974% that a three-way meeting has occurred by tick 20.

## Python


In [1]:
from itertools import product

track_size = 12
starts = (0, 4, 8)
ticks = 20

# Keep probabilities only for states where the tokens have not all met yet.
still_separate = {starts: 1.0}
probability_met_by_tick = []
met_so_far = 0.0

for tick in range(1, ticks + 1):
    next_states = {}
    met_this_tick = 0.0

    for positions, state_probability in still_separate.items():
        for moves in product((-1, 1), repeat=3):
            new_positions = tuple(
                (position + move) % track_size
                for position, move in zip(positions, moves)
            )
            outcome_probability = state_probability / 8

            if new_positions[0] == new_positions[1] == new_positions[2]:
                met_this_tick += outcome_probability
            else:
                next_states[new_positions] = (
                    next_states.get(new_positions, 0.0) + outcome_probability
                )

    still_separate = next_states
    met_so_far += met_this_tick
    probability_met_by_tick.append(met_so_far)

for tick in (1, 5, 10, 20):
    print(f"By tick {tick:2}: {probability_met_by_tick[tick - 1]:.4%}")

By tick  1: 0.0000%
By tick  5: 1.2451%
By tick 10: 7.7509%
By tick 20: 21.6870%


In [2]:
print("\nCumulative chance of a meeting (each # is about 2 percentage points):")
for tick, probability in enumerate(probability_met_by_tick, start=1):
    bar = "#" * round(probability * 50)
    print(f"{tick:2} | {bar} {probability:6.2%}")


Cumulative chance of a meeting (each # is about 2 percentage points):
 1 |   0.00%
 2 |   0.00%
 3 |   0.00%
 4 |   0.44%
 5 | #  1.25%
 6 | #  2.31%
 7 | ##  3.56%
 8 | ##  4.90%
 9 | ###  6.31%
10 | ####  7.75%
11 | #####  9.20%
12 | ##### 10.65%
13 | ###### 12.10%
14 | ####### 13.52%
15 | ####### 14.93%
16 | ######## 16.33%
17 | ######### 17.70%
18 | ########## 19.05%
19 | ########## 20.38%
20 | ########### 21.69%


## Python interpretation

The state-probability calculation reports a cumulative meeting chance of about **21.69% by tick 20**. Meeting probability is accumulated when a meeting first occurs, so this is the chance of at least one meeting?not merely the chance the tokens coincide at the final tick.